# Titanic Dataset Exploration

Report notebook for the Titanic preprocessing pipeline.

In [ ]:
import sys
from pathlib import Path

import joblib
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root / "src"))

from pipeline import TitanicPreprocessor, get_titanic_data, split_data

pre = TitanicPreprocessor()

## Section 1: Load Raw Data & Check Missing Values (Before Cleaning)

In [ ]:
raw_df = get_titanic_data(str(project_root / "data" / "raw" / "titanic.csv"))

raw_df.info()
print()
print(raw_df.isnull().sum())

## Section 2: Title Extraction

In [ ]:
df = pre.extract_title(raw_df.copy())
print(df["Title"].value_counts())

## Section 3: Cleaning & Feature Engineering

In [ ]:
df = pre.clean(df)
df = pre.feature_engineer(df)

df.head(5)

## Section 4: Missing Values After Cleaning

In [ ]:
missing_after_cleaning = df.isnull().sum()
print(missing_after_cleaning)
print()
print("Cabin removed:", "Cabin" not in df.columns)
print("Embarked missing values:", missing_after_cleaning["Embarked"])
print("Age missing values:", missing_after_cleaning["Age"])

## Section 5: Train-Validation-Test Split

In [ ]:
X_train, X_val, X_test = split_data(df)

print("Training Set:", X_train.shape)
print("Validation Set:", X_val.shape)
print("Testing Set:", X_test.shape)

## Section 6: Apply Pipeline & Save

In [ ]:
X_train_processed = pre.fit_transform(X_train)
X_val_processed = pre.transform(X_val)
X_test_processed = pre.transform(X_test)

print("Processed Training Set:", X_train_processed.shape)
print("Processed Validation Set:", X_val_processed.shape)
print("Processed Testing Set:", X_test_processed.shape)

joblib.dump(pre.preprocessor, project_root / "titanic_preprocessing_pipeline.pkl")
print("Pipeline Saved Successfully")

## Section 7: Results Summary

### Missing Values Handled

In [ ]:
before_missing = raw_df.isnull().sum()
after_missing = df.isnull().sum()

missing_values_handled = pd.DataFrame(
    {
        "Before": [
            before_missing["Age"],
            before_missing["Cabin"],
            before_missing["Embarked"],
        ],
        "After": [
            after_missing["Age"],
            "Removed",
            after_missing["Embarked"],
        ],
    },
    index=["Age", "Cabin", "Embarked"],
)

missing_values_handled

### Feature Engineering

| Feature | Purpose |
| --- | --- |
| FamilySize | Combines SibSp and Parch into total family size on board (plus the passenger). |
| IsAlone | Flags passengers traveling without family members. |
| Title | Captures social status and gender cues extracted from the passenger name. |

### Dataset Split

In [ ]:
dataset_split = pd.DataFrame(
    {
        "Split": ["Train", "Validation", "Test"],
        "Raw Shape": [X_train.shape, X_val.shape, X_test.shape],
        "Processed Shape": [
            X_train_processed.shape,
            X_val_processed.shape,
            X_test_processed.shape,
        ],
    }
)

dataset_split

### Completion Checklist

- [x] Data Cleaned
- [x] Features Engineered
- [x] Dataset Split
- [x] Pipeline Saved